# Cleaning Downloaded Data from avian-flu

Author: Alexander Maksiaev

Purpose: Clean downloaded data from avian-flu, rename sequences according to convention, de-duplicate from GISAID

In [1]:
# Housekeeping

import os
import glob 
import pandas as pd
import xml.etree.ElementTree as ET
import requests
import time
import numpy as np
import dateutil 
from datetime import datetime
from collections import defaultdict 
import importlib
import utils  
importlib.reload(utils)
from utils import * 

# Make sure you have the correct paths

# home = "C:/Users/maksi/Documents/Statistics/Projects/Avian_Flu_Files/"
home = "C:/Users/maksiaevai.NCBI_NT/Documents/Avian_Flu"
# downloads = "C:/Users/maksi/Documents/Statistics/Projects/Avian_Flu_Files/"
downloads = "C:/Users/maksiaevai.NCBI_NT/Documents/Avian_Flu_Files/"
originals = downloads + "Andersen_Downloads/"
temp_files = downloads + "Andersen_Temp_Files/"
complete_files = downloads + "Andersen_Complete_Files/"

# Day we're updating data
update_date = "04-21-2025"

os.chdir(downloads)

## Read Metadata 

In [2]:
# Read metadata

metadata_folder = originals + "avian-influenza/metadata/"
os.chdir(metadata_folder)

metadata = pd.read_csv("SraRunTable_automated.csv")

# print(len(metadata)) # 7397 rows

# Get rid of missing dates; they won't be counted anyway
for date in metadata["Collection_Date"]:
    if "/" in date or date == "missing":
        metadata = metadata[metadata["Collection_Date"] != date]

# Find only >= 2024 to start
metadata["Collection_Date_Compare"] = metadata["Collection_Date"].apply(lambda x: dateutil.parser.parse(x).strftime("%Y-%m-%d"))
metadata = metadata[metadata["Collection_Date_Compare"] >= datetime(2024, 1, 1).strftime("%Y-%m-%d")]

# Find only >= last date using Release Date from metadata 
metadata["ReleaseDate"] = metadata["ReleaseDate"].apply(lambda x: dateutil.parser.parse(x).strftime("%Y-%m-%d"))
metadata = metadata[metadata["ReleaseDate"] >= datetime(2024, 1, 1).strftime("%Y-%m-%d")]

print(len(metadata)) # 6053 rows between 1/1/2024 and 4/14/2025

7044


In [ ]:
# # Get list of genotypes

# os.chdir(home)

# genotypes_df = pd.read_excel("genotype_key.xlsx")

# genotypes = list(genotypes_df["Genotype"])

# print(genotypes)

### Naming convention ###
>A/[host]/[geo_loc_name]/[isolate]/[year]|[serotype: H5N1]|[collection_date]|[host_type]|[genotype]

host_type is from manual animal reference

In metadata, we have: host, geo_loc_name, isolate, year

We need: geo_loc_name, collection_date, host_type, genotype

host = Host

geo_loc_name (primary) = geo_loc_name

geo_loc_name (secondary) = genbank_mapping.tsv > genbank_name

isolate = isolate

collection date (primary) = Collection_Date

collection date (secondary) = https://www.ncbi.nlm.nih.gov/genbank/ > BioSample (input: BioSample) > Nucleotide > [first result] > collection_date

serotype = serotype

host type = [from ref] 

genotype = [from genoflu] -- use genoflu_results.tsv

## Get genotype, specific geolocation

In [4]:
# Get genotype from genoflu_results.tsv

os.chdir(metadata_folder)

genoflu_results = pd.read_csv("genoflu_results.tsv", delimiter="\t")

metadata["Genotype"] = genoflu_results["Genotype"]
metadata = metadata[~metadata["Genotype"].str.contains('Not assigned')] # Do not include non-assigned genotypes

# Get only the genotypes we want: B3.13 and D1.1

b313_and_d11_only = genoflu_results[(genoflu_results["Genotype"] == "B3.13") | (genoflu_results["Genotype"] == "D1.1")]
b313_and_d11_only = b313_and_d11_only.rename(columns={"sample": "Run"})
b313_and_d11_only = b313_and_d11_only.drop_duplicates(subset="Run", keep="last")

metadata = metadata.merge(b313_and_d11_only, on="Run", how="inner")

print(len(metadata)) 

5829


In [5]:
# Get specific geolocation from genbank_mapping.tsv

genbank_mapping = pd.read_csv("genbank_mapping.tsv", delimiter="\t")
genbank_mapping["Run"] = genbank_mapping["sra_run"]
genbank_mapping = genbank_mapping.drop_duplicates(subset="Run", keep="first") # Drop duplicates
genbank_mapping["name_state"] = genbank_mapping["genbank_name"].apply(lambda x: x.split("/")[2]) # Get the name of the state

metadata_genbank = metadata.merge(genbank_mapping, on=["Run"], how="inner") # Only include data that has states

print(genbank_mapping["name_state"])
print(len(metadata_genbank))
display(metadata_genbank)

0        Texas
8        Texas
16       Texas
24       Texas
32       Texas
         ...  
38678       OH
38686       OH
38694       OH
38702       OH
38710       MT
Name: name_state, Length: 4279, dtype: object
3692


,Run,Assay Type,AvgSpotLen,Bases,BioProject,BioSample,BioSampleModel,Bytes,Center Name,Collection_Date,...,Genotype Mismatch List,Genotype Average Depth of Coverage List,seg_file,seg_seq_name,sra_run,seg,genbank_acc,genbank_seg,genbank_name,name_state
0,SRR28752447,WGS,241.29,86080323,PRJNA1102327,SAMN41019237,Viral,28164109,USDA-NVSL,2024,...,"26, 10, 18, 21, 9, 13, 11, 6",Ran on FASTA - No Coverage Report,SRR28752447_HA_cns.fa,Consensus_SRR28752447_HA_cns_threshold_0.5_qua...,SRR28752447,HA,PP752829.1,4,A/cattle/Texas/24-009108-005/2024,Texas
1,SRR28752448,WGS,250.30,75035343,PRJNA1102327,SAMN41019236,Viral,24547283,USDA-NVSL,2024,...,"26, 10, 17, 21, 10, 13, 11, 7",Ran on FASTA - No Coverage Report,SRR28752448_HA_cns.fa,Consensus_SRR28752448_HA_cns_threshold_0.5_qua...,SRR28752448,HA,PP752821.1,4,A/cattle/Texas/24-009108-004/2024,Texas
2,SRR28752449,WGS,146.61,59363690,PRJNA1102327,SAMN41019235,Viral,19686302,USDA-NVSL,2024,...,"26, 10, 18, 21, 9, 13, 11, 6",Ran on FASTA - No Coverage Report,SRR28752449_HA_cns.fa,Consensus_SRR28752449_HA_cns_threshold_0.5_qua...,SRR28752449,HA,PP752813.1,4,A/cattle/Texas/24-009108-003/2024,Texas
3,SRR28752450,WGS,251.31,119232569,PRJNA1102327,SAMN41019234,Viral,38846827,USDA-NVSL,2024,...,"27, 10, 17, 21, 10, 13, 11, 7",Ran on FASTA - No Coverage Report,SRR28752450_HA_cns.fa,Consensus_SRR28752450_HA_cns_threshold_0.5_qua...,SRR28752450,HA,PP752805.1,4,A/cattle/Texas/24-009108-002/2024,Texas
4,SRR28752453,WGS,144.71,37055919,PRJNA1102327,SAMN41019231,Viral,12938409,USDA-NVSL,2024,...,"26, 12, 16, 24, 9, 12, 12, 7",Ran on FASTA - No Coverage Report,SRR28752453_HA_cns.fa,Consensus_SRR28752453_HA_cns_threshold_0.5_qua...,SRR28752453,HA,PP752677.1,4,A/cattle/Texas/24-009088-001/2024,Texas
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
3687,SRR32633088,WGS,145.11,67016467,PRJNA1102327,SAMN47290851,Viral,25388495,USDA-NVSL,2025,...,"32, 11, 21, 24, 17, 15, 11, 9",Ran on FASTA - No Coverage Report,SRR32633088_HA_cns.fa,Consensus_SRR32633088_HA_cns_threshold_0.5_qua...,SRR32633088,HA,PV456280.1,4,A/cat/OR/25-005913-003-original/2025,OR
3688,SRR32633089,WGS,148.14,140023511,PRJNA1102327,SAMN47290850,Viral,52157519,USDA-NVSL,2025,...,"33, 11, 22, 25, 16, 15, 11, 9",Ran on FASTA - No Coverage Report,SRR32633089_HA_cns.fa,Consensus_SRR32633089_HA_cns_threshold_0.5_qua...,SRR32633089,HA,PV456272.1,4,A/cat/OR/25-005800-002-original/2025,OR
3689,SRR32633090,WGS,148.49,83727737,PRJNA1102327,SAMN47290849,Viral,31619233,USDA-NVSL,2025,...,"33, 12, 23, 25, 16, 16, 11, 8",Ran on FASTA - No Coverage Report,SRR32633090_HA_cns.fa,Consensus_SRR32633090_HA_cns_threshold_0.5_qua...,SRR32633090,HA,PV456264.1,4,A/cat/OR/25-005800-001-original/2025,OR
3690,SRR32633093,WGS,148.10,105910536,PRJNA1102327,SAMN47290846,Viral,39757708,USDA-NVSL,2025,...,"32, 16, 23, 29, 18, 18, 10, 7",Ran on FASTA - No Coverage Report,SRR32633093_HA_cns.fa,Consensus_SRR32633093_HA_cns_threshold_0.5_qua...,SRR32633093,HA,PV457240.1,4,A/cattle/CA/25-005677-001-original/2025,CA


## Get and save collection date

In [6]:

# Get all dates
# metadata_genbank["Collection_Date_Specific"] = metadata_genbank["BioSample"].apply(lambda x: search_collection_date(x, metadata_genbank))

# Save this so we don't have to do it again

# os.chdir(temp_files)
# metadata_genbank.to_csv("metadata_genbank.csv")

In [ ]:
# Upload saved data -- if doing this, make sure the above cell is commented out
os.chdir(temp_files + "saved/")
metadata_genbank_dates = pd.read_csv("metadata_genbank_4-18-2025.csv")
os.chdir(temp_files)

# Get only updated dates

unknown_dates = metadata_genbank_dates[(metadata_genbank_dates["Collection_Date_Specific"] == "2024") | (metadata_genbank_dates["Collection_Date_Specific"] == "2025")] # Dates we don't have
known_dates = metadata_genbank_dates[(metadata_genbank_dates["Collection_Date_Specific"] != "2024") & (metadata_genbank_dates["Collection_Date_Specific"] != "2025")] # Dates we've already gotten

# Get new dates also 
# new_dates = metadata_genbank["BioSample"].apply(lambda x: search_collection_date(x, metadata_genbank) if )

updated_unknown_dates = unknown_dates["BioSample"].apply(lambda x: search_collection_date(x, unknown_dates)) # Update unknown dates, if possible

metadata_genbank = pd.concat([known_dates, unknown_dates], ignore_index=True, sort=True)


display(metadata_genbank)

SAMN41019181
Unable to find collection date.
SAMN41106834
Unable to find collection date.
SAMN41106833
Unable to find collection date.
SAMN41106832
Unable to find collection date.
SAMN41106831
Unable to find collection date.
SAMN41106830
Unable to find collection date.
SAMN41106829
Unable to find collection date.
SAMN41106828
Unable to find collection date.
SAMN41106827
Unable to find collection date.
SAMN41106826
Unable to find collection date.
SAMN41106825
Unable to find collection date.
SAMN41106783
Unable to find collection date.
SAMN41106780
Unable to find collection date.
SAMN41106820
Unable to find collection date.
SAMN41106819
Unable to find collection date.
SAMN41106815
Unable to find collection date.
SAMN41106808
Unable to find collection date.
SAMN41106755
Unable to find collection date.
SAMN41489080
Unable to find collection date.
SAMN41489095
Unable to find collection date.
SAMN41489094
Unable to find collection date.
SAMN41489093
Unable to find collection date.
SAMN414890

,Assay Type,AvgSpotLen,Bases,BioProject,BioSample,BioSample Accession,BioSampleModel,Bytes,Center Name,Collection_Date,...,isolate,isolation_source,name_state,retraction_detection_date_utc,seg,seg_file,seg_seq_name,serotype,sra_run,version
0,WGS,241.29,86080323,PRJNA1102327,SAMN41019237,SRS21079811,Viral,28164109,USDA-NVSL,2024,...,24-009108-005-original,NaN,Texas,NaN,HA,SRR28752447_HA_cns.fa,Consensus_SRR28752447_HA_cns_threshold_0.5_qua...,H5N1,SRR28752447,1
1,WGS,250.30,75035343,PRJNA1102327,SAMN41019236,SRS21079810,Viral,24547283,USDA-NVSL,2024,...,24-009108-004-original,NaN,Texas,NaN,HA,SRR28752448_HA_cns.fa,Consensus_SRR28752448_HA_cns_threshold_0.5_qua...,H5N1,SRR28752448,1
2,WGS,146.61,59363690,PRJNA1102327,SAMN41019235,SRS21079809,Viral,19686302,USDA-NVSL,2024,...,24-009108-003-original,NaN,Texas,NaN,HA,SRR28752449_HA_cns.fa,Consensus_SRR28752449_HA_cns_threshold_0.5_qua...,H5N1,SRR28752449,1
3,WGS,251.31,119232569,PRJNA1102327,SAMN41019234,SRS21079808,Viral,38846827,USDA-NVSL,2024,...,24-009108-002-original,NaN,Texas,NaN,HA,SRR28752450_HA_cns.fa,Consensus_SRR28752450_HA_cns_threshold_0.5_qua...,H5N1,SRR28752450,1
4,WGS,144.71,37055919,PRJNA1102327,SAMN41019231,SRS21079806,Viral,12938409,USDA-NVSL,2024,...,24-009088-001-original,NaN,Texas,NaN,HA,SRR28752453_HA_cns.fa,Consensus_SRR28752453_HA_cns_threshold_0.5_qua...,H5N1,SRR28752453,1
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
4058,WGS,146.98,106556038,PRJNA1102327,SAMN46921959,SRS24134530,Viral,39746392,USDA-NVSL,2025,...,25-003807-006,food sample,CO,NaN,HA,SRR32415259_HA_cns.fa,Consensus_SRR32415259_HA_cns_threshold_0.5_qua...,NaN,SRR32415259,1
4059,WGS,144.20,18361062,PRJNA1102327,SAMN46921958,SRS24134528,Viral,6990746,USDA-NVSL,2025,...,25-003807-005,food sample,CO,NaN,HA,SRR32415260_HA_cns.fa,Consensus_SRR32415260_HA_cns_threshold_0.5_qua...,NaN,SRR32415260,1
4060,WGS,146.80,49731175,PRJNA1102327,SAMN46921956,SRS24134526,Viral,18679978,USDA-NVSL,2025,...,25-003807-003,food sample,CO,NaN,HA,SRR32415262_HA_cns.fa,Consensus_SRR32415262_HA_cns_threshold_0.5_qua...,NaN,SRR32415262,1
4061,WGS,146.76,50267804,PRJNA1102327,SAMN46921955,SRS24134525,Viral,18729119,USDA-NVSL,2025,...,25-003807-002,food sample,CO,NaN,HA,SRR32415263_HA_cns.fa,Consensus_SRR32415263_HA_cns_threshold_0.5_qua...,NaN,SRR32415263,1


In [12]:
print(metadata_genbank[["Collection_Date_Specific"]])

     Collection_Date_Specific
0                 20-Mar-2024
1                 20-Mar-2024
2                 20-Mar-2024
3                 20-Mar-2024
4                 13-Mar-2024
...                       ...
4058                     2025
4059                     2025
4060                     2025
4061                     2025
4062                     2025

[4063 rows x 1 columns]


## Get host type

In [13]:
# Create animals ref if needed

unique_animals_all = sort_animals_andersen(metadata_genbank)

# Flatten unique_animals_all
every_unique_animal = []
for animal in unique_animals_all:
    every_unique_animal.append(animal)

print(every_unique_animal)

unique_animals_set = list(set(every_unique_animal)) # Get rid of duplicates

os.chdir(downloads)

animals_ref = pd.read_csv("animals_ref.csv") # Upload animals ref

# If animal not in ref1, put in ref2

common_animals = []
# Check if animals in unique_animals_set are in ref1
for animal in unique_animals_set:
    for col in animals_ref.columns:
        if animal in animals_ref[col].values and type(animal) == str:
            common_animals.append(animal)

# If not in ref1, make a list of the new animals
different_animals = []
for animal in unique_animals_set:
    if animal not in common_animals:
        different_animals.append(animal)

print(different_animals)

# Add to dataframe
animals_df = animals_ref
# Make different_animals same length as dataframe, if shorter
if len(different_animals) < len(animals_df):
    number_of_times_to_add_nan = len(animals_df) - len(different_animals)
    for i in range(number_of_times_to_add_nan):
        different_animals.append(float('nan'))
# If longer, deal with that later

animals_df["new"] = (different_animals)

print(animals_df)

animals_df.to_csv("animals_ref_to_sort.csv") # Make sure name is different to avoid overwriting the first reference 


['barn owl', 'hosp', 'cat', 'common raven', 'quail', 'bottlenose dolphin', 'ganada goose', 'hawk', 'eurasian collared dove', 'western sandpiper', 'rock pigeon', 'cattle milk product', 'cago', 'red-shouldered hawk', 'pefa', 'merganser', 'serval', 'hooded merganser', 'lion', "cooper's hawk", 'savannah cat', 'wood duck', 'black-crowned night-heron', 'vulture', 'falcon', 'owl', 'fox', 'great horned owl', 'american crow', 'raccoon', 'pheasant', 'dove', 'domestic cat', 'red tailed hawk', 'pig', 'emu', 'gadwall', 'peafowl', 'cackling goose', 'blackbird', 'snow goose', "geoffroy's cat", 'red fox', 'great blue heron', 'american wigeon', 'goat', 'burrowing owl', 'western gull', 'swan', 'crane', 'guinea fowl', 'goose', 'house-mouse', 'avian', 'skunk', 'pet food', 'lynx', 'tiger', 'grackle', 'green-winged teal', 'american robin', 'snowy owl', 'pigeon', 'domestic-cat', 'raw pet food', 'ermine', 'red-breasted merganser', 'bald eagle', 'chicken', 'mountain lion', 'turkey vulture', 'house mouse', 'com

In [14]:
# Get animals from animal reference
os.chdir(downloads)
animals_ref = pd.read_csv("animals_ref.csv")
fix_animals_andersen(metadata_genbank, animals_ref) # Get host type

metadata_genbank["years"] = metadata_genbank["Collection_Date"].apply(lambda x: str(x).split("-")[0]) # Get year only from collection date

## Make names using all the attributes we collected

In [15]:
for num, collection_date in enumerate(metadata_genbank["Collection_Date_Specific"]):
    if collection_date != collection_date: # If nan
        metadata_genbank.loc[num, "Collection_Date_Specific"] = metadata_genbank.loc[num, "years"]
    else: # If actual date
        if len(str(collection_date)) == 4: # If it's a year
            # print("caught")
            metadata_genbank.loc[num, "Collection_Date_Specific"] = collection_date
        else:
            parsed_date = dateutil.parser.parse(collection_date)
            date = parsed_date.strftime("%Y-%m-%d") # Make sure it doesn't default to today, if just a year
            metadata_genbank.loc[num, "Collection_Date_Specific"] = date

# Make names

names = ">A/" + metadata_genbank["Host"] + "/" + metadata_genbank["name_state"] + "/" + metadata_genbank["isolate"] + "/" + metadata_genbank["years"].apply(lambda x: str(x)) + "|H5N1|" + metadata_genbank["Collection_Date_Specific"].apply(lambda x: str(x)) + "|" + metadata_genbank["Host_Type"] + "|" + metadata_genbank["Genotype"]

metadata_genbank["Name"] = names

# metadata_genbank.to_csv("metadata_genbank_named.csv")

# display(metadata_genbank)

## Make FASTA files

In [16]:
# Get information to create the fasta files

fasta_folder = originals + "avian-influenza/fasta/"

os.chdir(fasta_folder)

segments = ["PB2", "PB1", "PA", "NS", "NP", "NA", "MP", "HA"]
pairs = []
fasta_files = {}

for genotype in ["B3.13", "D1.1"]:
    for segment in segments:
        pair = genotype + "_" + segment
        pairs.append(pair)

for pair in pairs:
    fasta_files[pair] = [] # List to hold fasta files

for run in metadata_genbank["Run"].values: # For each run 
    for dirpath, dirs, files in os.walk(fasta_folder): # Find the fasta file
        for file in files:
            file_name = os.path.join(dirpath, file) # Get file name
            # print(file_name)
            if run in file_name: # Note that there will be ~8 files total with that run name
                # Make a fasta file and put it in the list
                with open(file_name) as f:
                    lines = f.readlines()
                    sequence = lines[1] 
                    # Each run/segment pair has one sequence -- it's placed into a file with other run/segment pairs with the same segment and genotype
                    header = metadata_genbank[metadata_genbank["Run"] == run].loc[:, "Name"].values[0]
                    genotype = metadata_genbank[metadata_genbank["Run"] == run].loc[:, "Genotype"].values[0]
                    # print(header)
                    # print(genotype)
                    # break 
                    segment = file_name.split("_")[-2]
                    # Find the pair that corresponds to 
                    pair_name = genotype + "_" + segment
                    this_specific_fasta = []
                    for pair in pairs:
                        # print(pair)
                        # print(pair_name)
                        if pair_name == pair:
                            this_specific_fasta.append(header)
                            this_specific_fasta.append(sequence)
                            fasta_files[pair].append(this_specific_fasta)
                f.close()

D1.3
D1.3
D1.3
D1.3
D1.3
D1.3
D1.3
D1.3
B3.13
B3.13
B3.13
B3.13
B3.13
B3.13
B3.13
B3.13
B3.13
B3.13
B3.13
B3.13
B3.13
B3.13
B3.13
B3.13
B3.13
B3.13
B3.13
B3.13
B3.13
B3.13
B3.13
B3.13
B3.13
B3.13
B3.13
B3.13
B3.13
B3.13
B3.13
B3.13
D1.1
D1.1
D1.1
D1.1
D1.1
D1.1
D1.1
D1.1
B3.13
B3.13
B3.13
B3.13
B3.13
B3.13
B3.13
B3.13
A1
A1
A1
A1
A1
A1
A1
A1
B3.6
B3.6
B3.6
B3.6
B3.6
B3.6
B3.6
B3.6
B3.13
B3.13
B3.13
B3.13
B3.13
B3.13
B3.13
B3.13
B3.13
B3.13
B3.13
B3.13
B3.13
B3.13
B3.13
B3.13
B3.13
B3.13
B3.13
B3.13
B3.13
B3.13
B3.13
B3.13
B3.13
B3.13
B3.13
B3.13
B3.13
B3.13
B3.13
B3.13
B3.13
B3.13
B3.13
B3.13
B3.13
B3.13
B3.13
B3.13
B3.13
B3.13
B3.13
B3.13
B3.13
B3.13
B3.13
B3.13
D1.1
D1.1
D1.1
D1.1
D1.1
D1.1
D1.1
D1.1
B3.13
B3.13
B3.13
B3.13
B3.13
B3.13
B3.13
B3.13
B3.13
B3.13
B3.13
B3.13
B3.13
B3.13
B3.13
B3.13
B3.13
B3.13
B3.13
B3.13
B3.13
B3.13
B3.13
B3.13
D1.3
D1.3
D1.3
D1.3
D1.3
D1.3
D1.3
D1.3
B1.1
B1.1
B1.1
B1.1
B1.1
B1.1
B1.1
B1.1
C2.1
C2.1
C2.1
C2.1
C2.1
C2.1
C2.1
C2.1
D1.1
D1.1
D1.1
D1.1
D1.1

In [17]:
# Create fasta files 

os.chdir(temp_files)

for pair in fasta_files.keys():
    output_path = temp_files + pair + "_andersen_" + update_date + ".fasta" 

    output_file = open(output_path, "w")
    for item in fasta_files[pair]:
        # for item in item:
        # item = fasta_files[pair]
        try:
            name = str(item[0].values[0]) # See if this is one we didn't have a collection date for
        except:
            name = str(item[0])
        print(name)
        # First is header, second is sequence
        # print(value)
        output_file.write(name + "\n")
        output_file.write(item[1])
    output_file.close()

>A/Cattle/Texas/24-009108-004-original/2024|H5N1|2024-03-20|cattle|B3.13
>A/Cattle/Texas/24-009108-003-original/2024|H5N1|2024-03-20|cattle|B3.13
>A/Cattle/Texas/24-009108-002-original/2024|H5N1|2024-03-20|cattle|B3.13
>A/Cattle/Texas/24-009088-001-original/2024|H5N1|2024-03-13|cattle|B3.13
>A/Cattle/Texas/24-009029-001-original/2024|H5N1|2024-03-21|cattle|B3.13
>A/Cattle/Texas/24-009028-009-original/2024|H5N1|2024-03-20|cattle|B3.13
>A/Cattle/Texas/24-009028-008-original/2024|H5N1|2024-03-20|cattle|B3.13
>A/Cattle/Texas/24-009028-007-original/2024|H5N1|2024-03-20|cattle|B3.13
>A/Cattle/Texas/24-009028-005-original-repeat/2024|H5N1|2024-03-20|cattle|B3.13
>A/Cattle/Texas/24-009028-002-original/2024|H5N1|2024-03-20|cattle|B3.13
>A/Cattle/Texas/24-009028-001-original/2024|H5N1|2024-03-20|cattle|B3.13
>A/Cattle/Michigan/24-009027-005-original-MTM/2024|H5N1|2024-03-24|cattle|B3.13
>A/Cattle/Michigan/24-009027-004-original-repeat/2024|H5N1|2024-03-24|cattle|B3.13
>A/Chicken/Texas/24-007264-

## De-Duplication

In [2]:
# De-duplication 

# Gisaid 

gisaid = downloads + "GISAID_Complete_Fasta_Files/04-01-2025--04-14-2025_renamed/"

os.chdir(gisaid)

dfs_gisaid = create_dataframes(gisaid)

In [7]:
for key in dfs_gisaid.keys():
    dataframes = dfs_gisaid[key]
    print(dataframes)

[    isolate_partial                                        full_header  \
0                P-  >A/dairy_cow/Idaho/W241290019-18-P/2024|H5N1|2...   
1                P-  >A/dairy_cow/Idaho/W241290019-19-P/2024|H5N1|2...   
2      W241220059-8  >A/dairy_cow/Idaho/W241220059-8/2024|H5N1|2024...   
3      W241220059-9  >A/dairy_cow/Idaho/W241220059-9/2024|H5N1|2024...   
4     W240870066-26  >A/dairy_cow/Idaho/W240870066-26/2024|H5N1|202...   
..              ...                                                ...   
202      000600-002  >A/dairy_cow/California/25_000600-002/2024|H5N...   
203      000590-002  >A/dairy_cow/California/25_000590-002/2024|H5N...   
204      005209-001  >A/dairy_cow/California/25_005209-001/2024|H5N...   
205      004920-005  >A/dairy_cow/California/25_004920-005/2024|H5N...   
206      004920-004  >A/dairy_cow/California/25_004920-004/2024|H5N...   

                                              sequence  
0    atggagaacatagtactacttcttgcaatagttagccttgttaaaa..

In [3]:
# Do the same with Andersen 

dfs_andersen = create_dataframes(temp_files)

In [4]:
for key in dfs_andersen.keys():
    dataframes = dfs_andersen[key]
    print(dataframes)

[     isolate_partial                                        full_header  \
0         009108-004  >A/Cattle/Texas/24-009108-004-original/2024|H5...   
1         009108-003  >A/Cattle/Texas/24-009108-003-original/2024|H5...   
2         009108-002  >A/Cattle/Texas/24-009108-002-original/2024|H5...   
3         009088-001  >A/Cattle/Texas/24-009088-001-original/2024|H5...   
4         009029-001  >A/Cattle/Texas/24-009029-001-original/2024|H5...   
...              ...                                                ...   
2138      003807-010  >A/PET FOOD/CO/25-003807-010/2025|H5N1|2025|ot...   
2139      003807-009  >A/PET FOOD/CO/25-003807-009/2025|H5N1|2025|ot...   
2140      003807-008  >A/PET FOOD/CO/25-003807-008/2025|H5N1|2025|ot...   
2141      003807-005  >A/PET FOOD/CO/25-003807-005/2025|H5N1|2025|ot...   
2142      003807-003  >A/PET FOOD/CO/25-003807-003/2025|H5N1|2025|ot...   

                                               sequence  
0     ATGGAGAACATAGTACTACTTCTTGCAATAGTTA

In [ ]:
# Merge dataframes and drop duplicates

full_dfs = defaultdict(list)

# for key in dfs_andersen.keys():
#     dataframes = dfs_andersen[key]
#     for i, df in enumerate(dataframes):
#         print(i)
#         try:
#             full_df = df.merge(dfs_gisaid[key][i], how="outer")
#             # print(full_df)
#             full_df = full_df.drop_duplicates(subset=["isolate_partial"])
#             full_dfs[key].append(full_df)
#         except:
#             print("Failed to merge dataframes in ", key)



0
Failed to merge dataframes in  D1.1_PB2_andersen_04-21-2025


In [ ]:
# If none in one database, only use the other and drop duplicates

full_dfs = defaultdict(list)
for key in dfs_andersen.keys():
    print(key)
# for key in ["D1.3"]:
    dataframes = dfs_andersen[key]
    for i, df in enumerate(dataframes):
        print(i)
        try:
            # full_df = df.merge(dfs_gisaid[key][i], how="outer")
            # print(full_df)
            full_df = full_df.drop_duplicates(subset=["isolate_partial"])
            full_dfs[key].append(full_df)
        except:
            print("Failed to merge dataframes in ", key)

A1_HA_andersen_2025-04-14
0
Failed to merge dataframes in  A1_HA_andersen_2025-04-14
A1_MP_andersen_2025-04-14
0
Failed to merge dataframes in  A1_MP_andersen_2025-04-14
A1_NA_andersen_2025-04-14
0
Failed to merge dataframes in  A1_NA_andersen_2025-04-14
A1_NP_andersen_2025-04-14
0
Failed to merge dataframes in  A1_NP_andersen_2025-04-14
A1_NS_andersen_2025-04-14
0
Failed to merge dataframes in  A1_NS_andersen_2025-04-14
A1_PA_andersen_2025-04-14
0
Failed to merge dataframes in  A1_PA_andersen_2025-04-14
A1_PB1_andersen_2025-04-14
0
Failed to merge dataframes in  A1_PB1_andersen_2025-04-14
A1_PB2_andersen_2025-04-14
0
Failed to merge dataframes in  A1_PB2_andersen_2025-04-14
A2_HA_andersen_2025-04-14
0
Failed to merge dataframes in  A2_HA_andersen_2025-04-14
A2_MP_andersen_2025-04-14
0
Failed to merge dataframes in  A2_MP_andersen_2025-04-14
A2_NA_andersen_2025-04-14
0
Failed to merge dataframes in  A2_NA_andersen_2025-04-14
A2_NP_andersen_2025-04-14
0
Failed to merge dataframes in  A2

## Create FASTA files combining Andersen and GISAID

In [ ]:
# # Create fasta files 

# os.chdir(complete_files)
# for pair in full_dfs.keys():
#     output_path = complete_files + pair + "_combined_" + update_date + ".fasta" 

#     output_file = open(output_path, "w")
#     for item in full_dfs[pair]:
#         # for item in item:
#         # item = fasta_files[pair]
#         for index, row in item.iterrows():
#             name = item.loc[index, "full_header"]
#             sequence = item.loc[index, "sequence"]
#         # print(name)
#         # First is header, second is sequence
#         # print(value)
#             output_file.write(name)
#             output_file.write(sequence)
#     output_file.close()

In [ ]:
# os.chdir(complete_files)

# segment_files = []
# for dirpath, dirs, files in os.walk(temp_files): # + "01-01-2024--03-31-2025_renamed/"):
#     for segment in ["PB2", "PB1", "PA", "NS", "NP", "NA", "MP", "HA"]:
#         base = []
#         for file in files:
#             file_name = os.path.join(dirpath, file)
#             # print(file_name)
#             if segment in file_name:
#                 with open(file_name) as f:
#                     lines = f.readlines()
#                     for line in lines:
#                         base.append(line)
#                 f.close()
#         os.chdir(complete_files)
#         output_path = complete_files + segment + "_andersen_" + update_date + ".fasta" # Genotype and Segment should all be the same
#         output_file = open(output_path, "w")
#         for b in base:
#             output_file.write(b)
#         output_file.close()